# Data Engineering: from the raw CSV files to the final dataset
**IT3091 Machine Learning · Group 2026-DS-09 · European football match outcome prediction (H / D / A)**

This notebook builds the dataset used by our ML pipeline **from the original raw files**, one step at a time, and then proves the result is correct with independent checks.

```
csv/stage_1/  European Soccer database (SQLite tables exported to CSV)   2008/09 - 2015/16
csv/stage_2/  Transfermarkt database (CSV bundle)                        2016/17 - 2025/26
      |                                   |
  PART A: Stage 1 steps               PART B: Stage 2 steps
  (same rules, same 62 columns)       (same rules, same 62 columns)
      |                                   |
      +------------ PART C: stack rows -------------+
                         |
            Master (11 leagues, 55,585 matches)
                         |   drop Switzerland (C1) and Poland (PL1)
            Primary (9 leagues, 51,201 matches)  -->  ML pipeline
```

| Part | What it does |
|---|---|
| 0 | Setup: Colab or Jupyter, paths, the rules (constants) |
| A | Stage 1: load raw tables, target, starting XI and roles, form, player strength, team features |
| B | Stage 2: the same steps on the Transfermarkt files |
| C | Combine: checks before combining, stack into Master, filter to Primary, save |
| D | Walk through one real match by hand |
| E | Independent verification: recompute features with simple slow code and compare |

**Leakage rule used everywhere:** every feature for a match uses only information available *before kickoff*. Goals are kept only to create the target and are never model inputs.

## 0. Setup
### 0.1 Where are the raw files?
* **Google Colab:** put the `csv` folder (with `stage_1` and `stage_2` inside) in your Google Drive and set `DRIVE_FOLDER` below. Then run *Runtime > Run all*.
* **Jupyter on your computer:** put this notebook next to the `csv` folder, or edit `STAGE1_DIR` / `STAGE2_DIR`.

Only these raw files are read: Stage 1 = `Match`, `League`, `country`, `Team`, `Player Attributes`. Stage 2 = `games`, `competitions`, `game_lineups`, `player_valuations`.

In [1]:
import os, re, time
from bisect import bisect_left, bisect_right, insort
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

IN_COLAB = "google.colab" in str(globals().get("get_ipython", lambda: "")())
DRIVE_FOLDER = "/content/drive/MyDrive/Machine Learning/Assignment"   # <- edit for Colab

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path(DRIVE_FOLDER)
else:
    BASE = Path(os.environ.get("DE_BASE", "."))                      # folder that contains csv/

STAGE1_DIR = BASE / "csv" / "stage_1"
STAGE2_DIR = BASE / "csv" / "stage_2"
OUTPUT_DIR = BASE / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Optional: engineered files to compare against in Part E (leave as None if you do not have them)
EXISTING_PRIMARY = os.environ.get("DE_EXISTING_PRIMARY")   # e.g. BASE / "European_Football_PlayerAware_Primary_9Leagues_2008_2026.csv"

print("pandas", pd.__version__, "| numpy", np.__version__, "| Colab:", IN_COLAB)
print("Stage 1 folder:", STAGE1_DIR, "exists:", STAGE1_DIR.exists())
print("Stage 2 folder:", STAGE2_DIR, "exists:", STAGE2_DIR.exists())
T0 = time.time()

pandas 3.0.5 | numpy 2.5.3 | Colab: False
Stage 1 folder: csv/stage_1 exists: True
Stage 2 folder: csv/stage_2 exists: True


### 0.2 The rules (all thresholds in one place)
| Constant | Value | Meaning |
|---|---|---|
| `FORM_WINDOW` | 5 | recent form = the previous 5 league matches of the same season |
| `MAX_VALUE_AGE_DAYS` | 730 | a FIFA rating or market value older than 2 years is not used |
| `MIN_REFERENCE_N` | 30 | a percentile is given only when at least 30 earlier values exist for that league and role |
| `MIN_KNOWN_FOR_XI` | 8 | team XI strength needs at least 8 starters with a value |
| `MIN_KNOWN_FOR_TOP3` | 3 | top-3 strength needs at least 3 starters with a value |
| `MIN_COVERAGE_READY` | 0.70 | `player_feature_ready = 1` when both XIs have 11 starters and at least 70% have a value |
| `REFERENCE_FIRST_SEASON_STAGE2` | 2012 | Transfermarkt lineups from 2012-2015 are used only as a warm-up reference for percentiles |

In [2]:
FORM_WINDOW = 5
MAX_VALUE_AGE_DAYS = 730
MIN_REFERENCE_N = 30
MIN_KNOWN_FOR_XI = 8
MIN_KNOWN_FOR_TOP3 = 3
MIN_COVERAGE_READY = 0.70
REFERENCE_FIRST_SEASON_STAGE2 = 2012
FULL_XI = 11
ROUND_DECIMALS = 4
Y_GK_MAX, Y_DEF_MAX, Y_MID_MAX = 1, 4, 8      # Stage 1 vertical pitch coordinate bands

STAGE1_LEAGUE_CODES = {
    "Belgium Jupiler League": "BE1", "England Premier League": "GB1", "France Ligue 1": "FR1",
    "Germany 1. Bundesliga": "L1", "Italy Serie A": "IT1", "Netherlands Eredivisie": "NL1",
    "Poland Ekstraklasa": "PL1", "Portugal Liga ZON Sagres": "PO1", "Scotland Premier League": "SC1",
    "Spain LIGA BBVA": "ES1", "Switzerland Super League": "C1",
}
ALL_LEAGUES = ["BE1", "ES1", "FR1", "GB1", "IT1", "L1", "NL1", "PL1", "PO1", "SC1", "C1"]
PRIMARY_LEAGUES = ["BE1", "ES1", "FR1", "GB1", "IT1", "L1", "NL1", "PO1", "SC1"]

STAGE1_SOURCE, STAGE1_METHOD = "european_soccer_sqlite_stage1", "fifa_overall_rating_causal_prior_league_role_percentile"
STAGE2_SOURCE, STAGE2_METHOD = "transfermarkt_stage2", "market_value_log_causal_prior_league_role_percentile"

# The common 62-column schema shared by both stages
COMMON_COLUMNS = [
    "match_uid", "data_source", "strength_method", "game_id", "date", "season_start", "season",
    "league_code", "league_name", "country", "league_coverage_flag",
    "home_team_id", "away_team_id", "home_team", "away_team", "home_formation", "away_formation",
    "home_form_n_L5", "away_form_n_L5", "home_points_per_game_L5", "away_points_per_game_L5",
    "home_win_rate_L5", "away_win_rate_L5", "home_goals_for_avg_L5", "away_goals_for_avg_L5",
    "home_goals_against_avg_L5", "away_goals_against_avg_L5", "home_goal_diff_avg_L5", "away_goal_diff_avg_L5",
    "home_starter_count", "away_starter_count", "home_strength_known_n", "away_strength_known_n",
    "home_strength_coverage", "away_strength_coverage", "home_strength_age_mean_days", "away_strength_age_mean_days",
    "home_xi_strength", "away_xi_strength", "home_attack_strength", "away_attack_strength",
    "home_midfield_strength", "away_midfield_strength", "home_defence_strength", "away_defence_strength",
    "home_gk_strength", "away_gk_strength", "home_top3_strength", "away_top3_strength",
    "home_top3_gap", "away_top3_gap", "home_lineup_continuity", "away_lineup_continuity",
    "xi_strength_diff", "home_attack_vs_away_defence", "away_attack_vs_home_defence",
    "midfield_strength_diff", "gk_strength_diff", "player_feature_ready",
    "audit_home_goals", "audit_away_goals", "target_result",
]
print(len(COMMON_COLUMNS), "columns in the common schema")

62 columns in the common schema


### 0.3 Helper functions used by both stages
Each function does one job. They are shared, so **Stage 1 and Stage 2 follow exactly the same rules**.

* `add_form_features`: recent form from the previous 5 matches (current match excluded, restarts each season)
* `add_lineup_continuity`: share of today's XI that also started the previous match
* `attach_latest_value`: for each starter, the latest rating or market value dated **strictly before** the match
* `causal_percentile`: turns a rating or value into a 0-100 rank within league and role, using **only earlier dates**
* `aggregate_side_features`: player values to team values (XI, attack, midfield, defence, goalkeeper, top-3)

In [3]:
def _norm(text):
    return re.sub(r"[^a-z0-9]", "", text.lower())

def find_csv(folder, table_name):
    # find '<table_name>.csv' ignoring case, spaces and underscores (so 'Player Attributes' = 'Player_Attributes')
    for p in Path(folder).iterdir():
        if p.suffix.lower() == ".csv" and _norm(p.stem) == _norm(table_name):
            return p
    raise FileNotFoundError(f"No CSV like '{table_name}' in {folder}")

def read_columns(path, needed, **kw):
    header = pd.read_csv(path, nrows=0).columns.tolist()
    missing = [c for c in needed if c not in header]
    if missing:
        raise KeyError(f"{Path(path).name} is missing {missing}")
    return pd.read_csv(path, usecols=needed, **kw)

def to_ns(s):
    return pd.to_datetime(s, errors="coerce").astype("datetime64[ns]")

def result_from_goals(home, away):
    return pd.Series(np.where(home > away, "H", np.where(home == away, "D", "A")), index=home.index)

def _is_missing(x):
    return x is None or (isinstance(x, float) and x != x) or x is pd.NA

In [4]:
def add_form_features(matches, window=FORM_WINDOW):
    # one row per team per match, sorted in time
    home = matches[["game_id", "date", "season_start", "home_team_id", "home_goals", "away_goals"]].rename(
        columns={"home_team_id": "team_id", "home_goals": "gf", "away_goals": "ga"}); home["side"] = "home"
    away = matches[["game_id", "date", "season_start", "away_team_id", "away_goals", "home_goals"]].rename(
        columns={"away_team_id": "team_id", "away_goals": "gf", "home_goals": "ga"}); away["side"] = "away"
    long = pd.concat([home, away], ignore_index=True).sort_values(["team_id", "season_start", "date", "game_id"]).reset_index(drop=True)
    long["points"] = np.where(long.gf > long.ga, 3, np.where(long.gf == long.ga, 1, 0))
    long["win"] = (long.gf > long.ga).astype(int)
    long["gd"] = long.gf - long.ga
    g = long.groupby(["team_id", "season_start"], sort=False)
    # shift(1) = drop the current match, so only EARLIER matches are averaged
    prior = lambda col: g[col].transform(lambda s: s.shift(1).rolling(window, min_periods=1).mean())
    long["points_per_game_L5"] = prior("points"); long["win_rate_L5"] = prior("win")
    long["goals_for_avg_L5"] = prior("gf"); long["goals_against_avg_L5"] = prior("ga"); long["goal_diff_avg_L5"] = prior("gd")
    long["form_n_L5"] = g["points"].transform(lambda s: s.shift(1).rolling(window, min_periods=1).count()).fillna(0).astype(int)
    cols = ["form_n_L5", "points_per_game_L5", "win_rate_L5", "goals_for_avg_L5", "goals_against_avg_L5", "goal_diff_avg_L5"]
    out = matches[["game_id"]].copy()
    for side in ("home", "away"):
        part = long.loc[long.side == side, ["game_id"] + cols].copy()
        part.columns = ["game_id"] + [f"{side}_{c}" for c in cols]
        out = out.merge(part, on="game_id", how="left")
    for c in out.columns:
        if c != "game_id" and not c.endswith("form_n_L5"):
            out[c] = out[c].round(ROUND_DECIMALS)
    return out


def add_lineup_continuity(matches, starters):
    # share of the current XI that also started the team's previous same-season match with recorded starters
    lineups = starters.groupby(["game_id", "side"])["player_id"].agg(frozenset)
    rows = []
    for side in ("home", "away"):
        part = matches[["game_id", "date", "season_start", f"{side}_team_id"]].rename(columns={f"{side}_team_id": "team_id"})
        part["side"] = side; rows.append(part)
    long = pd.concat(rows, ignore_index=True)
    long["lineup"] = [lineups.get((g, s), frozenset()) for g, s in zip(long.game_id, long.side)]
    long = long.sort_values(["team_id", "season_start", "date", "game_id"]).reset_index(drop=True)
    cont = np.full(len(long), np.nan); pos = 0
    for _, grp in long.groupby(["team_id", "season_start"], sort=False):
        prev = None
        for lineup in grp["lineup"]:
            if prev is not None and len(lineup) == FULL_XI and len(prev) == FULL_XI:
                cont[pos] = len(lineup & prev) / FULL_XI
            if len(lineup) > 0:
                prev = lineup
            pos += 1
    long["continuity"] = cont
    out = matches[["game_id"]].copy()
    for side in ("home", "away"):
        part = long.loc[long.side == side, ["game_id", "continuity"]].rename(columns={"continuity": f"{side}_lineup_continuity"})
        out = out.merge(part, on="game_id", how="left")
        out[f"{side}_lineup_continuity"] = out[f"{side}_lineup_continuity"].round(ROUND_DECIMALS)
    return out

In [5]:
def attach_latest_value(starters, values, id_col, value_date_col, value_col, max_age_days=MAX_VALUE_AGE_DAYS):
    # merge_asof 'backward' + allow_exact_matches=False  ->  latest value dated STRICTLY BEFORE the match day
    left = starters.copy(); left["date"] = to_ns(left["date"]); left[id_col] = left[id_col].astype("int64")
    right = values[[id_col, value_date_col, value_col]].dropna().copy()
    right[value_date_col] = to_ns(right[value_date_col]); right[id_col] = right[id_col].astype("int64")
    merged = pd.merge_asof(left.sort_values("date", kind="stable"), right.sort_values(value_date_col, kind="stable"),
                           left_on="date", right_on=value_date_col, by=id_col, direction="backward",
                           allow_exact_matches=False, tolerance=pd.Timedelta(days=max_age_days))
    merged["value_age_days"] = (merged["date"] - merged[value_date_col]).dt.days.astype(float)
    return merged   # keeps value_date_col so the no-future rule can be checked in Part E


def causal_percentile(starters, value_col="raw_value"):
    # 0-100 rank of each player among EARLIER starters in the same league and role.
    #   mid_rank = (#below + #equal / 2);   percentile = 100 * (mid_rank - 0.5) / (n - 1)
    # All matches on one date are scored first, and only then added to the reference.
    ordered = starters.sort_values("date", kind="stable")
    dates = ordered["date"].to_numpy(); leagues = ordered["league_code"].to_numpy(dtype=object)
    roles = ordered["role"].to_numpy(dtype=object); values = ordered[value_col].to_numpy(dtype=float)
    result = np.full(len(ordered), np.nan); reference = {}; n = len(ordered); i = 0
    while i < n:
        j = i
        while j < n and dates[j] == dates[i]:
            j += 1
        for k in range(i, j):                       # 1) score every starter on this date
            if np.isnan(values[k]) or _is_missing(roles[k]):
                continue
            ref = reference.get((leagues[k], roles[k]))
            if ref is not None and len(ref) >= MIN_REFERENCE_N:
                lo, hi = bisect_left(ref, values[k]), bisect_right(ref, values[k])
                result[k] = min(100.0, max(0.0, 100.0 * ((lo + hi) / 2.0 - 0.5) / (len(ref) - 1)))
        for k in range(i, j):                       # 2) only now add them to the reference
            if np.isnan(values[k]) or _is_missing(roles[k]):
                continue
            insort(reference.setdefault((leagues[k], roles[k]), []), values[k])
        i = j
    return pd.Series(result, index=ordered.index).reindex(starters.index)


def aggregate_side_features(starters):
    key = ["game_id", "side"]; s = starters.copy()
    s["value_age_days"] = s["value_age_days"].where(s["strength"].notna())
    g = s.groupby(key)
    f = pd.DataFrame({"starter_count": g["player_id"].size(), "strength_known_n": g["strength"].count(),
                      "strength_age_mean_days": g["value_age_days"].mean(), "xi_strength": g["strength"].mean()})
    f["xi_strength"] = f["xi_strength"].where(f["strength_known_n"] >= MIN_KNOWN_FOR_XI)
    f["strength_coverage"] = f["strength_known_n"] / f["starter_count"]
    for role, name in [("ATT", "attack_strength"), ("MID", "midfield_strength"), ("DEF", "defence_strength"), ("GK", "gk_strength")]:
        f[name] = s[s["role"] == role].groupby(key)["strength"].mean()
    top = s[s["strength"].notna()].sort_values("strength", ascending=False).groupby(key).head(3).groupby(key)["strength"].agg(["mean", "count"])
    f["top3_strength"] = top["mean"].where(top["count"] >= MIN_KNOWN_FOR_TOP3)
    f["top3_gap"] = f["top3_strength"] - f["xi_strength"]
    return f.reset_index()


def side_features_to_wide(side_feats, game_ids):
    metrics = [c for c in side_feats.columns if c not in ("game_id", "side")]
    out = pd.DataFrame({"game_id": game_ids.to_numpy()})
    for side in ("home", "away"):
        part = side_feats.loc[side_feats.side == side, ["game_id"] + metrics].copy()
        part.columns = ["game_id"] + [f"{side}_{c}" for c in metrics]
        out = out.merge(part, on="game_id", how="left")
        for c in (f"{side}_starter_count", f"{side}_strength_known_n"):
            out[c] = out[c].fillna(0).astype(int)
    for c in out.columns:
        if c != "game_id" and not c.endswith(("_starter_count", "_strength_known_n")):
            out[c] = out[c].round(ROUND_DECIMALS)
    return out


def add_matchup_features(df):
    df = df.copy()
    df["xi_strength_diff"] = df.home_xi_strength - df.away_xi_strength
    df["home_attack_vs_away_defence"] = df.home_attack_strength - df.away_defence_strength
    df["away_attack_vs_home_defence"] = df.away_attack_strength - df.home_defence_strength
    df["midfield_strength_diff"] = df.home_midfield_strength - df.away_midfield_strength
    df["gk_strength_diff"] = df.home_gk_strength - df.away_gk_strength
    for c in ["xi_strength_diff", "home_attack_vs_away_defence", "away_attack_vs_home_defence", "midfield_strength_diff", "gk_strength_diff"]:
        df[c] = df[c].round(ROUND_DECIMALS)
    return df


def add_player_feature_ready(df):
    df = df.copy()
    df["player_feature_ready"] = ((df.home_starter_count == 11) & (df.away_starter_count == 11) &
                                  (df.home_strength_coverage >= MIN_COVERAGE_READY) &
                                  (df.away_strength_coverage >= MIN_COVERAGE_READY)).astype(int)
    return df


def finalise(df):
    df = df.copy(); df["date"] = pd.to_datetime(df["date"]).dt.strftime("%Y-%m-%d")
    missing = [c for c in COMMON_COLUMNS if c not in df.columns]
    assert not missing, f"missing columns: {missing}"
    return df[COMMON_COLUMNS].sort_values(["date", "league_code", "match_uid"]).reset_index(drop=True)

print("Helper functions ready.")

Helper functions ready.


---
# PART A: Stage 1 (European Soccer database, 2008/09 to 2015/16)
### A1. Load the raw tables
We read only the columns we need. The shot, card, possession and bookmaker-odds columns in `Match` are **not** read: events happen during the match (leakage), and odds are outside our research question.

In [6]:
home_ids = [f"home_player_{i}" for i in range(1, 12)]; away_ids = [f"away_player_{i}" for i in range(1, 12)]
home_y = [f"home_player_Y{i}" for i in range(1, 12)];  away_y = [f"away_player_Y{i}" for i in range(1, 12)]

s1_match = read_columns(find_csv(STAGE1_DIR, "Match"),
    ["match_api_id", "league_id", "season", "date", "home_team_api_id", "away_team_api_id",
     "home_team_goal", "away_team_goal"] + home_ids + away_ids + home_y + away_y)
s1_league = read_columns(find_csv(STAGE1_DIR, "League"), ["id", "country_id", "name"])
s1_country = read_columns(find_csv(STAGE1_DIR, "country"), ["id", "name"])
s1_team = read_columns(find_csv(STAGE1_DIR, "Team"), ["team_api_id", "team_long_name"])
s1_ratings = read_columns(find_csv(STAGE1_DIR, "Player Attributes"), ["player_api_id", "date", "overall_rating"])

for name, df in [("Match", s1_match), ("League", s1_league), ("country", s1_country), ("Team", s1_team), ("Player Attributes", s1_ratings)]:
    print(f"{name:18s} {df.shape[0]:>8,} rows x {df.shape[1]} columns")
s1_match[["match_api_id", "league_id", "season", "date", "home_team_api_id", "away_team_api_id",
          "home_team_goal", "away_team_goal", "home_player_1", "home_player_Y1", "home_player_Y11"]].head()

Match                25,979 rows x 52 columns
League                   11 rows x 3 columns
country                  11 rows x 2 columns
Team                    299 rows x 2 columns
Player Attributes   183,978 rows x 3 columns


,match_api_id,league_id,season,date,home_team_api_id,away_team_api_id,home_team_goal,away_team_goal,home_player_1,home_player_Y1,home_player_Y11
0,492473,1,2008/2009,2008-08-17 00:00:00,9987,9993,1,1,NaN,NaN,NaN
1,492474,1,2008/2009,2008-08-16 00:00:00,10000,9994,0,0,NaN,NaN,NaN
2,492475,1,2008/2009,2008-08-16 00:00:00,9984,8635,0,3,NaN,NaN,NaN
3,492476,1,2008/2009,2008-08-17 00:00:00,9991,9998,5,0,NaN,NaN,NaN
4,492477,1,2008/2009,2008-08-16 00:00:00,7947,9985,1,3,NaN,NaN,NaN


### A2. Match list: league codes, team names, season, unique ID
League names are mapped to short codes (for example `England Premier League` -> `GB1`) so they match the Transfermarkt codes in Stage 2. Each match gets `match_uid = "S1_" + match_api_id`, so it can never clash with a Transfermarkt ID.

In [7]:
lg = s1_league.rename(columns={"id": "league_id", "name": "league_name"}).merge(
     s1_country.rename(columns={"id": "country_id", "name": "country"}), on="country_id", how="left")
lg["league_code"] = lg["league_name"].map(STAGE1_LEAGUE_CODES)
display(lg[["league_id", "league_name", "country", "league_code"]])

m = s1_match.merge(lg[["league_id", "league_name", "country", "league_code"]], on="league_id", how="left")
m = m[m.league_code.notna()].copy()
names = s1_team.set_index("team_api_id")["team_long_name"]
m["date"] = to_ns(m["date"]).dt.normalize()
m = m[m.home_team_goal.notna() & m.away_team_goal.notna()].reset_index(drop=True)

s1_matches = pd.DataFrame({
    "game_id": m.match_api_id.astype("int64"),
    "match_uid": "S1_" + m.match_api_id.astype("int64").astype(str),
    "data_source": STAGE1_SOURCE, "strength_method": STAGE1_METHOD,
    "date": m.date, "season": m.season, "season_start": m.season.str[:4].astype(int),
    "league_code": m.league_code, "league_name": m.league_name, "country": m.country,
    "league_coverage_flag": "full_2008_2015",
    "home_team_id": m.home_team_api_id.astype("int64"), "away_team_id": m.away_team_api_id.astype("int64"),
    "home_team": m.home_team_api_id.map(names), "away_team": m.away_team_api_id.map(names),
    "home_goals": m.home_team_goal.astype(int), "away_goals": m.away_team_goal.astype(int),
})
print("Stage 1 matches:", len(s1_matches))
display(pd.crosstab(s1_matches.league_code, s1_matches.season))

,league_id,league_name,country,league_code
0,1,Belgium Jupiler League,Belgium,BE1
1,1729,England Premier League,England,GB1
2,4769,France Ligue 1,France,FR1
3,7809,Germany 1. Bundesliga,Germany,L1
4,10257,Italy Serie A,Italy,IT1
5,13274,Netherlands Eredivisie,Netherlands,NL1
6,15722,Poland Ekstraklasa,Poland,PL1
7,17642,Portugal Liga ZON Sagres,Portugal,PO1
8,19694,Scotland Premier League,Scotland,SC1
9,21518,Spain LIGA BBVA,Spain,ES1


Stage 1 matches: 25979


season,2008/2009,2009/2010,2010/2011,2011/2012,2012/2013,2013/2014,2014/2015,2015/2016
league_code,,,,,,,,
BE1,306,210,240,240,240,12,240,240
C1,180,180,180,162,180,180,180,180
ES1,380,380,380,380,380,380,380,380
FR1,380,380,380,380,380,380,380,380
GB1,380,380,380,380,380,380,380,380
IT1,380,380,380,358,380,380,379,380
L1,306,306,306,306,306,306,306,306
NL1,306,306,306,306,306,306,306,306
PL1,240,240,240,240,240,240,240,240


### A3. Target variable from the final score
`H` = home win, `D` = draw, `A` = away win. The goals are kept as `audit_home_goals` / `audit_away_goals` so the target can be checked, but **they are never used as model inputs**.

In [8]:
s1_matches["target_result"] = result_from_goals(s1_matches.home_goals, s1_matches.away_goals)
print(s1_matches.target_result.value_counts().to_string())
s1_matches[["match_uid", "date", "home_team", "away_team", "home_goals", "away_goals", "target_result"]].head()

target_result
H    11917
A     7466
D     6596


,match_uid,date,home_team,away_team,home_goals,away_goals,target_result
0,S1_492473,2008-08-17,KRC Genk,Beerschot AC,1,1,D
1,S1_492474,2008-08-16,SV Zulte-Waregem,Sporting Lokeren,0,0,D
2,S1_492475,2008-08-16,KSV Cercle Brugge,RSC Anderlecht,0,3,A
3,S1_492476,2008-08-17,KAA Gent,RAEC Mons,5,0,H
4,S1_492477,2008-08-16,FCV Dender EH,Standard de Liège,1,3,A


### A4. Starting XI and each player's role
The `Match` table has 11 player-ID columns and 11 vertical pitch coordinates (`Y`) per team. We turn them into **one row per starter** and give a role from `Y`:

| Y | Role |
|---|---|
| 1 | GK (goalkeeper) |
| 2-4 | DEF |
| 5-8 | MID |
| 9 and up | ATT |

The formation label (for example `4-4-2`) is the count of DEF-MID-ATT from the same coordinates.

In [9]:
def role_from_y(y):
    y = pd.to_numeric(y, errors="coerce")
    role = pd.Series(np.select([y <= Y_GK_MAX, y <= Y_DEF_MAX, y <= Y_MID_MAX, y > Y_MID_MAX],
                               ["GK", "DEF", "MID", "ATT"], default=""), index=y.index, dtype=object)
    role[y.isna() | (role == "")] = None
    return role

parts = []
for side, id_cols, y_cols in (("home", home_ids, home_y), ("away", away_ids, away_y)):
    for pid_col, y_col in zip(id_cols, y_cols):
        parts.append(pd.DataFrame({"game_id": s1_matches.game_id.to_numpy(), "date": s1_matches.date.to_numpy(),
                                   "league_code": s1_matches.league_code.to_numpy(), "side": side,
                                   "player_id": m[pid_col].to_numpy(), "y": m[y_col].to_numpy()}))
s1_starters = pd.concat(parts, ignore_index=True)
s1_starters = s1_starters[s1_starters.player_id.notna()].copy()
s1_starters["player_id"] = s1_starters.player_id.astype("int64")
s1_starters = s1_starters.drop_duplicates(["game_id", "side", "player_id"])
s1_starters["role"] = role_from_y(s1_starters.y)

def formation_from_coordinates(match, game_ids):
    out = pd.DataFrame({"game_id": game_ids.to_numpy()})
    for side in ("home", "away"):
        y = match[[f"{side}_player_Y{i}" for i in range(1, 12)]].to_numpy(dtype=float); ok_y = ~np.isnan(y)
        n_gk = ((y >= 1) & (y <= Y_GK_MAX) & ok_y).sum(1); n_def = ((y > Y_GK_MAX) & (y <= Y_DEF_MAX) & ok_y).sum(1)
        n_mid = ((y > Y_DEF_MAX) & (y <= Y_MID_MAX) & ok_y).sum(1); n_att = ((y > Y_MID_MAX) & ok_y).sum(1)
        ok = (ok_y.sum(1) == FULL_XI) & (n_gk == 1)
        label = pd.Series(n_def).astype(str) + "-" + pd.Series(n_mid).astype(str) + "-" + pd.Series(n_att).astype(str)
        out[f"{side}_formation"] = label.where(pd.Series(ok)).to_numpy()
    return out
s1_formation = formation_from_coordinates(m, s1_matches.game_id)

print(f"Starter rows: {len(s1_starters):,}")
print(s1_starters.role.value_counts(dropna=False).to_string())
print("\nMost common formations:"); print(s1_formation.home_formation.value_counts().head(5).to_string())

EXAMPLE_S1 = s1_matches[(s1_matches.league_code == "GB1") & (s1_matches.season == "2015/2016")].iloc[0]
print(f"\nExample match: {EXAMPLE_S1.home_team} vs {EXAMPLE_S1.away_team}, {EXAMPLE_S1.date.date()}")
s1_starters[(s1_starters.game_id == EXAMPLE_S1.game_id) & (s1_starters.side == "home")].sort_values("y")

Starter rows: 542,280
role
MID     199348
DEF     189680
ATT      90531
GK       48058
None     14663

Most common formations:
home_formation
4-4-2    8790
4-5-1    8246
4-3-3    5427
3-5-2     821
3-4-3     594

Example match: Arsenal vs West Ham United, 2015-08-09


,game_id,date,league_code,side,player_id,y,role
4388,1987032,2015-08-09,GB1,home,30859,1.0,GK
30367,1987032,2015-08-09,GB1,home,26154,3.0,DEF
56346,1987032,2015-08-09,GB1,home,35606,3.0,DEF
82325,1987032,2015-08-09,GB1,home,46539,3.0,DEF
108304,1987032,2015-08-09,GB1,home,38521,3.0,DEF
134283,1987032,2015-08-09,GB1,home,159594,6.0,MID
160262,1987032,2015-08-09,GB1,home,37436,6.0,MID
186241,1987032,2015-08-09,GB1,home,75489,8.0,MID
212220,1987032,2015-08-09,GB1,home,36378,8.0,MID
238199,1987032,2015-08-09,GB1,home,196386,8.0,MID


### A5. Latest FIFA rating dated strictly before the match
For each starter we take the most recent `overall_rating` with a date **before** the match day (`allow_exact_matches=False`) and at most 730 days old. A rating published after the match can never be used.

In [10]:
r = s1_ratings.rename(columns={"player_api_id": "player_id", "date": "rating_date"})
s1_starters = attach_latest_value(s1_starters, r[r.overall_rating.notna()], id_col="player_id",
                                  value_date_col="rating_date", value_col="overall_rating")
s1_starters["raw_value"] = s1_starters["overall_rating"]
print(f"Starters with a rating: {s1_starters.raw_value.notna().mean():.1%}")
print("Rating date always before match date:", bool((s1_starters.rating_date.dropna() < s1_starters.loc[s1_starters.rating_date.notna(), 'date']).all()))
s1_starters[s1_starters.game_id == EXAMPLE_S1.game_id][["side", "player_id", "role", "date", "rating_date", "value_age_days", "overall_rating"]].head(6)

Starters with a rating: 98.9%
Rating date always before match date: True


,side,player_id,role,date,rating_date,value_age_days,overall_rating
471456,home,30859,GK,2015-08-09,2014-09-18,325.0,85.0
471474,home,26154,DEF,2015-08-09,2014-12-19,233.0,80.0
471492,home,35606,DEF,2015-08-09,2015-03-27,135.0,82.0
471510,home,46539,DEF,2015-08-09,2014-12-19,233.0,82.0
471528,home,38521,DEF,2015-08-09,2015-04-10,121.0,77.0
471546,home,159594,MID,2015-08-09,2015-04-24,107.0,75.0


### A6. Player strength as a causal percentile (0-100)
A rating of 80 means different things for a goalkeeper and a striker, and later we must also compare it with Transfermarkt market values. So each value is turned into a **rank among earlier starters in the same league and role**:

$$\text{percentile} = 100 \times \frac{\text{below} + \text{equal}/2 - 0.5}{n - 1}$$

where `below` and `equal` count reference values lower than and equal to this player, and `n` is the reference size (at least 30). Only matches on **earlier dates** are in the reference.

In [11]:
t = time.time()
s1_starters["strength"] = causal_percentile(s1_starters, "raw_value")
print(f"Percentiles computed in {time.time() - t:.0f}s | starters with a percentile: {s1_starters.strength.notna().mean():.1%}")
print(s1_starters.strength.describe().round(2).to_string())
s1_starters[s1_starters.game_id == EXAMPLE_S1.game_id][["side", "role", "overall_rating", "strength"]].head(11)

Percentiles computed in 4s | starters with a percentile: 95.9%
count    520031.00
mean         51.04
std          28.04
min           0.00
25%          27.13
50%          51.14
75%          74.80
max         100.00


,side,role,overall_rating,strength
471456,home,GK,85.0,96.145189
471474,home,DEF,80.0,82.741467
471492,home,DEF,82.0,90.886371
471510,home,DEF,82.0,90.886371
471528,home,DEF,77.0,64.252392
471546,home,MID,75.0,43.613211
471564,home,MID,85.0,94.941858
471582,home,MID,82.0,87.188840
471600,home,MID,86.0,96.814343
471618,home,MID,78.0,66.049874


### A7. Team features: strength, form, lineup continuity, matchups, ready flag

In [12]:
s1_side = side_features_to_wide(aggregate_side_features(s1_starters), s1_matches.game_id)
s1_form = add_form_features(s1_matches)
s1_cont = add_lineup_continuity(s1_matches, s1_starters)

stage1 = s1_matches.copy()
for block in (s1_form, s1_side, s1_cont, s1_formation):
    stage1 = stage1.merge(block, on="game_id", how="left")
stage1 = add_player_feature_ready(add_matchup_features(stage1))
stage1["audit_home_goals"], stage1["audit_away_goals"] = stage1.home_goals, stage1.away_goals
stage1 = finalise(stage1)
print("Stage 1 engineered:", stage1.shape)
stage1[stage1.game_id == EXAMPLE_S1.game_id][["home_team", "away_team", "home_points_per_game_L5", "home_xi_strength",
       "away_xi_strength", "xi_strength_diff", "home_lineup_continuity", "player_feature_ready", "target_result"]]

Stage 1 engineered: (25979, 62)


,home_team,away_team,home_points_per_game_L5,home_xi_strength,away_xi_strength,xi_strength_diff,home_lineup_continuity,player_feature_ready,target_result
22754,Arsenal,West Ham United,NaN,80.1791,41.0818,39.0973,NaN,1,A


### A8. Save Stage 1 and look at its missing values

In [13]:
stage1.to_csv(OUTPUT_DIR / "Stage1_PlayerAware_Football_2008_2016_Causal_v2.csv", index=False)
miss = stage1.isna().mean().mul(100).round(2)
print("Saved Stage 1:", stage1.shape)
print("Columns with gaps:", (miss > 0).sum(), "| largest gaps (%):"); print(miss.sort_values(ascending=False).head(6).to_string())

Saved Stage 1: (25979, 62)
Columns with gaps: 37 | largest gaps (%):
home_lineup_continuity    17.24
away_lineup_continuity    17.18
gk_strength_diff           9.94
away_gk_strength           8.92
home_gk_strength           8.83
xi_strength_diff           8.81


---
# PART B: Stage 2 (Transfermarkt, 2016/17 to 2025/26)
The steps are the same as Part A. Only three things differ, because the source is different:
1. Roles come from the **position text** in `game_lineups` (there are no pitch coordinates).
2. Strength comes from the **market value** (`log1p` of euros), not the FIFA rating.
3. Lineups from **2012-2015** are loaded only as a warm-up reference for the percentile, so the first 2016 matches already have enough earlier values. They do not become output rows.

### B1. Load the raw tables

In [14]:
s2_games = read_columns(find_csv(STAGE2_DIR, "games"),
    ["game_id", "competition_id", "season", "date", "home_club_id", "away_club_id", "home_club_goals",
     "away_club_goals", "home_club_formation", "away_club_formation", "home_club_name", "away_club_name"])
s2_comps = read_columns(find_csv(STAGE2_DIR, "competitions"), ["competition_id", "name", "country_name"])
s2_values = read_columns(find_csv(STAGE2_DIR, "player_valuations"), ["player_id", "date", "market_value_in_eur"])
for name, df in [("games", s2_games), ("competitions", s2_comps), ("player_valuations", s2_values)]:
    print(f"{name:18s} {df.shape[0]:>8,} rows x {df.shape[1]} columns")
s2_games.head(3)

games                88,958 rows x 12 columns
competitions             65 rows x 3 columns
player_valuations   656,301 rows x 3 columns


,game_id,competition_id,season,date,home_club_id,away_club_id,home_club_goals,away_club_goals,home_club_formation,away_club_formation,home_club_name,away_club_name
0,1026846,FIWC,2009,2010-06-26,3449,3589,2,1,NaN,NaN,Uruguay,South Korea
1,1026847,FIWC,2009,2010-06-27,3437,6303,3,1,NaN,NaN,Argentina,Mexico
2,1027001,FIWC,2009,2010-06-26,3505,3441,1,2,NaN,NaN,United States,Ghana


### B2. League matches, season, unique ID
`games` contains cups and other competitions too; only the 11 domestic leagues are kept. `match_uid = "TM_" + game_id`.

In [15]:
FIRST_SEASON_STAGE2 = 2016
g_all = s2_games[s2_games.competition_id.isin(ALL_LEAGUES) & (s2_games.season >= REFERENCE_FIRST_SEASON_STAGE2)].copy()
g_all = g_all[g_all.home_club_goals.notna() & g_all.away_club_goals.notna()]
g_all = g_all.merge(s2_comps.rename(columns={"name": "league_name", "country_name": "country"}), on="competition_id", how="left")
g_all["date"] = to_ns(g_all.date).dt.normalize(); g_all = g_all.reset_index(drop=True)
g = g_all[g_all.season >= FIRST_SEASON_STAGE2].reset_index(drop=True)
first_seen = g.groupby("competition_id")["season"].transform("min")

s2_matches = pd.DataFrame({
    "game_id": g.game_id.astype("int64"), "match_uid": "TM_" + g.game_id.astype("int64").astype(str),
    "data_source": STAGE2_SOURCE, "strength_method": STAGE2_METHOD,
    "date": g.date, "season": g.season.astype(int).map(lambda y: f"{y}/{str(y + 1)[-2:]}"),
    "season_start": g.season.astype(int), "league_code": g.competition_id,
    "league_name": g.league_name, "country": g.country,
    "league_coverage_flag": np.where(first_seen >= 2024, "partial_2024_2025", "full_2016_2025"),
    "home_team_id": g.home_club_id.astype("int64"), "away_team_id": g.away_club_id.astype("int64"),
    "home_team": g.home_club_name, "away_team": g.away_club_name,
    "home_goals": g.home_club_goals.astype(int), "away_goals": g.away_club_goals.astype(int),
    "home_formation": g.home_club_formation, "away_formation": g.away_club_formation,
})
s2_matches["target_result"] = result_from_goals(s2_matches.home_goals, s2_matches.away_goals)
print("Output matches:", len(s2_matches), "| matches in the percentile reference incl. warm-up:", len(g_all))
display(pd.crosstab(s2_matches.league_code, s2_matches.season_start))
print("Note: Switzerland (C1) and Poland (PL1) exist only from 2024 in this source; see Part C.")

Output matches: 29606 | matches in the percentile reference incl. warm-up: 41098


season_start,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
league_code,,,,,,,,,,
BE1,240,240,240,232,306,306,306,240,240,240
C1,0,0,0,0,0,0,0,0,202,228
ES1,380,380,380,380,380,380,380,380,380,380
FR1,380,380,380,279,380,380,380,306,306,306
GB1,380,380,380,380,380,380,380,380,380,380
IT1,380,380,380,380,380,380,380,380,380,380
L1,306,306,306,306,306,306,306,306,306,306
NL1,306,306,306,232,306,306,306,306,306,306
PL1,0,0,0,0,0,0,0,0,306,306


Note: Switzerland (C1) and Poland (PL1) exist only from 2024 in this source; see Part C.


### B3. Starting lineups and roles from position text
`game_lineups` is large (about 400 MB), so it is read in chunks and only starters of our league games are kept. Role rule (order matters, so a *Defensive Midfield* player is a midfielder):
`keeper` -> GK, `mid` -> MID, `back / defender / sweeper` -> DEF, everything else -> ATT.

In [16]:
def role_from_position(position):
    text = position.astype("string").str.lower()
    role = pd.Series(pd.NA, index=position.index, dtype="object"); known = text.notna()
    role[known] = "ATT"
    role[known & text.str.contains("back|defender|sweeper", na=False)] = "DEF"
    role[known & text.str.contains("mid", na=False)] = "MID"
    role[known & text.str.contains("keeper", na=False)] = "GK"
    return role

wanted = set(g_all.game_id); kept = []
for chunk in pd.read_csv(find_csv(STAGE2_DIR, "game_lineups"), usecols=["game_id", "club_id", "type", "player_id", "position"], chunksize=500_000):
    kept.append(chunk[(chunk["type"] == "starting_lineup") & chunk.game_id.isin(wanted)])
lineups = pd.concat(kept, ignore_index=True)
sides = pd.DataFrame({"game_id": g_all.game_id.astype("int64"), "date": g_all.date, "league_code": g_all.competition_id,
                      "home_team_id": g_all.home_club_id.astype("int64"), "away_team_id": g_all.away_club_id.astype("int64")})
s2_st = lineups.merge(sides, on="game_id", how="inner")
s2_st["side"] = np.where(s2_st.club_id == s2_st.home_team_id, "home", np.where(s2_st.club_id == s2_st.away_team_id, "away", ""))
s2_st = s2_st[s2_st.side != ""].dropna(subset=["player_id"]).copy()
s2_st["player_id"] = s2_st.player_id.astype("int64")
s2_st = s2_st.drop_duplicates(["game_id", "side", "player_id"])
s2_st["role"] = role_from_position(s2_st.position)
print(f"Starter rows (incl. warm-up): {len(s2_st):,}")
print("\nPosition text -> role (most common):")
display(s2_st.groupby(["position", "role"]).size().rename("starters").reset_index().sort_values("starters", ascending=False).head(14))
s2_starters_all = s2_st[["game_id", "date", "league_code", "side", "player_id", "role"]]

Starter rows (incl. warm-up): 837,310

Position text -> role (most common):


,position,role,starters
2,Centre-Back,DEF,170811
3,Centre-Forward,ATT,92325
1,Central Midfield,MID,84419
4,Defensive Midfield,MID,82517
5,Goalkeeper,GK,76121
11,Right-Back,DEF,60709
8,Left-Back,DEF,60692
0,Attacking Midfield,MID,48578
7,Left Winger,ATT,48086
10,Right Winger,ATT,47780


### B4. Latest market value strictly before the match, then the causal percentile
Market values are very skewed (a few players are worth over 100 million euros), so we use `log1p(value)`. The percentile turns it into the same 0-100 scale as Stage 1.

In [17]:
v = s2_values[s2_values.market_value_in_eur.notna()].rename(columns={"date": "valuation_date"})
s2_starters_all = attach_latest_value(s2_starters_all, v, id_col="player_id", value_date_col="valuation_date",
                                      value_col="market_value_in_eur")
s2_starters_all["raw_value"] = np.log1p(s2_starters_all.market_value_in_eur.astype(float))
t = time.time()
s2_starters_all["strength"] = causal_percentile(s2_starters_all, "raw_value")
print(f"Percentiles computed in {time.time() - t:.0f}s")
s2_starters = s2_starters_all[s2_starters_all.game_id.isin(set(s2_matches.game_id))].copy()   # output matches only
print(f"Starters in output matches: {len(s2_starters):,} | with a valuation: {s2_starters.raw_value.notna().mean():.1%} "
      f"| with a percentile: {s2_starters.strength.notna().mean():.1%}")
EXAMPLE_S2 = s2_matches[(s2_matches.league_code == "GB1") & (s2_matches.season_start == 2024)].sort_values("date").iloc[0]
print(f"\nExample match: {EXAMPLE_S2.home_team} vs {EXAMPLE_S2.away_team}, {EXAMPLE_S2.date.date()}")
s2_starters[(s2_starters.game_id == EXAMPLE_S2.game_id) & (s2_starters.side == "home")][
    ["role", "valuation_date", "value_age_days", "market_value_in_eur", "raw_value", "strength"]]

Percentiles computed in 7s
Starters in output matches: 647,166 | with a valuation: 99.2% | with a percentile: 99.2%

Example match: Manchester United vs Fulham FC, 2024-08-16


,role,valuation_date,value_age_days,market_value_in_eur,raw_value,strength
694552,MID,2024-05-27,81.0,20000000.0,16.811243,65.998417
694554,DEF,2024-05-27,81.0,18000000.0,16.705882,72.042202
694558,GK,2024-05-27,81.0,35000000.0,17.370859,90.116140
694559,ATT,2024-05-27,81.0,70000000.0,18.064006,92.364986
694562,ATT,2024-05-27,81.0,60000000.0,17.909855,89.122815
694564,DEF,2024-05-29,79.0,30000000.0,17.216708,88.365379
694565,MID,2024-05-27,81.0,35000000.0,17.370859,83.361440
694567,DEF,2024-05-27,81.0,35000000.0,17.370859,91.604214
694570,DEF,2024-07-19,28.0,50000000.0,17.727534,96.610664
694571,ATT,2024-05-27,81.0,18000000.0,16.705882,55.331366


### B5. Team features and save Stage 2

In [18]:
s2_side = side_features_to_wide(aggregate_side_features(s2_starters), s2_matches.game_id)
s2_form = add_form_features(s2_matches)
s2_cont = add_lineup_continuity(s2_matches, s2_starters)
stage2 = s2_matches.copy()
for block in (s2_form, s2_side, s2_cont):
    stage2 = stage2.merge(block, on="game_id", how="left")
stage2 = add_player_feature_ready(add_matchup_features(stage2))
stage2["audit_home_goals"], stage2["audit_away_goals"] = stage2.home_goals, stage2.away_goals
stage2 = finalise(stage2)
stage2.to_csv(OUTPUT_DIR / "Stage2_PlayerAware_Football_2016_2026_Causal_v2.csv", index=False)
miss = stage2.isna().mean().mul(100).round(2)
print("Saved Stage 2:", stage2.shape, "| columns with gaps:", (miss > 0).sum())
print(miss.sort_values(ascending=False).head(6).to_string())

Saved Stage 2: (29606, 62) | columns with gaps: 37
home_lineup_continuity       3.50
away_lineup_continuity       3.49
away_goals_against_avg_L5    2.84
home_points_per_game_L5      2.84
away_goals_for_avg_L5        2.84
home_goals_against_avg_L5    2.84


---
# PART C: Combine the two stages
### C1. Checks before combining
We **append rows** (stack). We do **not join** on IDs. The checks below show why that is safe and why a join would be wrong.

In [19]:
checks_c = {
    "Same 62 columns in the same order": list(stage1.columns) == list(stage2.columns) == COMMON_COLUMNS,
    "Stage 1 IDs all start with S1_": stage1.match_uid.str.startswith("S1_").all(),
    "Stage 2 IDs all start with TM_": stage2.match_uid.str.startswith("TM_").all(),
    "No match_uid in both stages": len(set(stage1.match_uid) & set(stage2.match_uid)) == 0,
    "Stage 1 ends before Stage 2 starts": stage1.date.max() < stage2.date.min(),
    "Same 0-100 strength scale": bool(stage1.home_xi_strength.dropna().between(0, 100).all()
                                      and stage2.home_xi_strength.dropna().between(0, 100).all()),
}
for k, val in checks_c.items():
    print(f"{'PASS' if val else 'FAIL'}  {k}")
print(f"\nLast Stage 1 match: {stage1.date.max()} | first Stage 2 match: {stage2.date.min()}")

# Why we never join on team ID: the same number can mean different clubs in the two databases
s1_names = pd.concat([stage1[["home_team_id", "home_team"]].set_axis(["id", "name"], axis=1)]).drop_duplicates("id").set_index("id")["name"]
s2_names = pd.concat([stage2[["home_team_id", "home_team"]].set_axis(["id", "name"], axis=1)]).drop_duplicates("id").set_index("id")["name"]
shared_ids = sorted(set(s1_names.index) & set(s2_names.index))
print("\nTeam ID numbers used in BOTH databases:", len(shared_ids))
display(pd.DataFrame({"team_id": shared_ids, "Stage 1 club": [s1_names[i] for i in shared_ids],
                      "Stage 2 club": [s2_names[i] for i in shared_ids]}))

PASS  Same 62 columns in the same order
PASS  Stage 1 IDs all start with S1_
PASS  Stage 2 IDs all start with TM_
PASS  No match_uid in both stages
PASS  Stage 1 ends before Stage 2 starts
PASS  Same 0-100 strength scale

Last Stage 1 match: 2016-05-25 | first Stage 2 match: 2016-07-29

Team ID numbers used in BOTH databases: 1


,team_id,Stage 1 club,Stage 2 club
0,8024,Widzew Łódź,FC Arouca


### C2. Master (all 11 leagues) and Primary (9 leagues)
Switzerland (`C1`) and Poland (`PL1`) are removed for the Primary dataset: Transfermarkt has them only from 2024/25, so their history has an eight-season gap, and many of their matches have no usable player data.

In [20]:
master = pd.concat([stage1, stage2], axis=0, ignore_index=True).sort_values(["date", "league_code", "match_uid"]).reset_index(drop=True)
assert not master.match_uid.duplicated().any(), "duplicate match_uid"
assert set(master.target_result.unique()) <= {"H", "D", "A"}
print("MASTER:", master.shape, "|", master.data_source.value_counts().to_dict())

weak = master[master.league_code.isin(["C1", "PL1"])]
print("\nSwitzerland and Poland: matches per season and source")
display(pd.crosstab(weak.data_source, weak.season_start))
print("Share of matches without usable player data (player_feature_ready = 0):")
print((1 - master.groupby("league_code").player_feature_ready.mean()).round(3).sort_values(ascending=False).to_string())

primary = master[master.league_code.isin(PRIMARY_LEAGUES)].sort_values(["date", "league_code", "match_uid"]).reset_index(drop=True)
print(f"\nRemoved {len(master) - len(primary):,} rows ({(len(master) - len(primary)) / len(master):.1%})")
print("PRIMARY:", primary.shape)
print(primary.target_result.value_counts().to_string())

MASTER: (55585, 62) | {'transfermarkt_stage2': 29606, 'european_soccer_sqlite_stage1': 25979}

Switzerland and Poland: matches per season and source


season_start,2008,2009,2010,2011,2012,2013,2014,2015,2024,2025
data_source,,,,,,,,,,
european_soccer_sqlite_stage1,420,420,420,402,420,420,420,420,0,0
transfermarkt_stage2,0,0,0,0,0,0,0,0,508,534


Share of matches without usable player data (player_feature_ready = 0):
league_code
PL1    0.626
C1     0.192
PO1    0.177
BE1    0.130
SC1    0.098
NL1    0.089
ES1    0.057
IT1    0.041
FR1    0.029
L1     0.014
GB1    0.013



Removed 4,384 rows (7.9%)
PRIMARY: (51201, 62)
target_result
H    23020
A    15408
D    12773


In [21]:
master.to_csv(OUTPUT_DIR / "European_Football_PlayerAware_Master_2008_2026.csv", index=False)
primary.to_csv(OUTPUT_DIR / "European_Football_PlayerAware_Primary_9Leagues_2008_2026.csv", index=False)
print("Saved to", OUTPUT_DIR)
for f in sorted(OUTPUT_DIR.glob("*.csv")):
    print(f"  {f.name:70s} {f.stat().st_size / 1e6:6.1f} MB")

Saved to output
  European_Football_PlayerAware_Master_2008_2026.csv                       25.8 MB
  European_Football_PlayerAware_Primary_9Leagues_2008_2026.csv             23.9 MB
  Stage1_PlayerAware_Football_2008_2016_Causal_v2.csv                      12.0 MB
  Stage2_PlayerAware_Football_2016_2026_Causal_v2.csv                      13.8 MB
  comparison_with_ML_dataset.csv                                            0.0 MB
  verification_checks.csv                                                   0.0 MB


---
# PART D: Walk through one match by hand
This recomputes the team strength of one real match with plain arithmetic, so you can see exactly where the numbers in the dataset come from.

In [22]:
def explain_match(example, starters, stage_df, value_label):
    row = stage_df[stage_df.game_id == example.game_id].iloc[0]
    print(f"{row.home_team} vs {row.away_team} | {row.date} | result {row.target_result} "
          f"({row.audit_home_goals}-{row.audit_away_goals})\n")
    for side in ("home", "away"):
        xi = starters[(starters.game_id == example.game_id) & (starters.side == side)]
        known = xi.strength.dropna()
        by_hand = known.mean() if len(known) >= MIN_KNOWN_FOR_XI else np.nan
        att = xi[xi.role == "ATT"].strength.mean(); gk = xi[xi.role == "GK"].strength.mean()
        print(f"{side.upper()} XI: {len(xi)} starters, {len(known)} with a percentile")
        display(xi[["role", "raw_value", "strength"]].rename(columns={"raw_value": value_label, "strength": "percentile"})
                .sort_values("role").round(2).reset_index(drop=True))
        print(f"  XI strength by hand = mean of {len(known)} percentiles = {by_hand:.4f} | in dataset = {row[f'{side}_xi_strength']}")
        print(f"  Attack strength by hand = {att:.4f} | in dataset = {row[f'{side}_attack_strength']}")
        print(f"  GK strength by hand = {gk:.4f} | in dataset = {row[f'{side}_gk_strength']}\n")
    print(f"xi_strength_diff = home - away = {row.home_xi_strength} - {row.away_xi_strength} = {row.xi_strength_diff}")

explain_match(EXAMPLE_S1, s1_starters, stage1, "FIFA rating")

Arsenal vs West Ham United | 2015-08-09 | result A (0-2)

HOME XI: 11 starters, 11 with a percentile


,role,FIFA rating,percentile
0,ATT,80.0,68.45
1,DEF,80.0,82.74
2,DEF,82.0,90.89
3,DEF,82.0,90.89
4,DEF,77.0,64.25
5,GK,85.0,96.15
6,MID,75.0,43.61
7,MID,85.0,94.94
8,MID,82.0,87.19
9,MID,86.0,96.81


  XI strength by hand = mean of 11 percentiles = 80.1791 | in dataset = 80.1791
  Attack strength by hand = 68.4503 | in dataset = 68.4503
  GK strength by hand = 96.1452 | in dataset = 96.1452

AWAY XI: 11 starters, 10 with a percentile


,role,FIFA rating,percentile
0,ATT,76.0,41.87
1,ATT,72.0,13.97
2,DEF,73.0,33.25
3,DEF,75.0,49.63
4,DEF,77.0,64.25
5,DEF,71.0,20.14
6,GK,77.0,41.12
7,MID,75.0,43.61
8,MID,NaN,NaN
9,MID,72.0,20.55


  XI strength by hand = mean of 10 percentiles = 41.0818 | in dataset = 41.0818
  Attack strength by hand = 27.9216 | in dataset = 27.9216
  GK strength by hand = 41.1180 | in dataset = 41.118

xi_strength_diff = home - away = 80.1791 - 41.0818 = 39.0973


In [23]:
explain_match(EXAMPLE_S2, s2_starters, stage2, "log1p(market value)")

Manchester United vs Fulham FC | 2024-08-16 | result H (1-0)

HOME XI: 11 starters, 11 with a percentile


,role,log1p(market value),percentile
0,ATT,18.06,92.36
1,ATT,17.91,89.12
2,ATT,16.71,55.33
3,DEF,16.71,72.04
4,DEF,17.22,88.37
5,DEF,17.37,91.60
6,DEF,17.73,96.61
7,GK,17.37,90.12
8,MID,16.81,66.00
9,MID,17.37,83.36


  XI strength by hand = mean of 11 percentiles = 83.4689 | in dataset = 83.4689
  Attack strength by hand = 78.9397 | in dataset = 78.9397
  GK strength by hand = 90.1161 | in dataset = 90.1161

AWAY XI: 11 starters, 11 with a percentile


,role,log1p(market value),percentile
0,ATT,16.01,32.20
1,ATT,17.03,66.93
2,ATT,16.81,59.53
3,DEF,16.52,65.46
4,DEF,16.81,76.55
5,DEF,17.03,83.58
6,DEF,16.59,68.14
7,GK,16.38,63.87
8,MID,16.81,66.00
9,MID,16.12,43.09


  XI strength by hand = mean of 11 percentiles = 63.1423 | in dataset = 63.1423
  Attack strength by hand = 52.8824 | in dataset = 52.8824
  GK strength by hand = 63.8709 | in dataset = 63.8709

xi_strength_diff = home - away = 83.4689 - 63.1423 = 20.3266


---
# PART E: Independent verification
The build code above is fast but not easy to read. Here each key feature is **recomputed with simple, slow code written separately**, on random samples, and compared with the dataset. If the build had a bug or used future information, these checks would fail.

| Check | What it proves |
|---|---|
| E1 | Recent form uses only the previous 5 same-season matches |
| E2 | Every percentile uses only earlier dates, same league and role |
| E3 | No rating or market value from the match day or later is used |
| E4 | Target, duplicates, ranges and the stage handover are correct |
| E5 | The rebuilt Primary file equals the file used by the ML pipeline |

In [24]:
rng = np.random.default_rng(42)
results = []
def record(name, passed, detail):
    results.append((name, "PASS" if passed else "FAIL", detail))
    print(f"{'PASS' if passed else 'FAIL'}  {name}: {detail}")

# E1: recent form by brute force
long_rows = []
for side in ("home", "away"):
    other = "away" if side == "home" else "home"
    long_rows.append(primary[["match_uid", "date", "season_start", f"{side}_team_id", f"audit_{side}_goals", f"audit_{other}_goals",
                              f"{side}_points_per_game_L5", f"{side}_goals_for_avg_L5"]]
                     .set_axis(["uid", "date", "season", "team", "gf", "ga", "ppg", "gfa"], axis=1))
L = pd.concat(long_rows, ignore_index=True)
L["pts"] = np.where(L.gf > L.ga, 3, np.where(L.gf == L.ga, 1, 0))
sample = L.sample(1000, random_state=42)
bad = 0
for _, r in sample.iterrows():
    prev = L[(L.team == r.team) & (L.season == r.season) & (L.date < r.date)].sort_values("date").tail(FORM_WINDOW)
    exp_ppg = round(prev.pts.mean(), 4) if len(prev) else np.nan
    exp_gf = round(prev.gf.mean(), 4) if len(prev) else np.nan
    if not ((np.isnan(exp_ppg) and np.isnan(r.ppg)) or (abs(exp_ppg - r.ppg) < 1e-6 and abs(exp_gf - r.gfa) < 1e-6)):
        bad += 1
record("E1 Recent form recomputed (1,000 random team-matches)", bad == 0, f"{1000 - bad} of 1000 identical")

PASS  E1 Recent form recomputed (1,000 random team-matches): 1000 of 1000 identical


In [25]:
# E2: percentile by brute force (reference = same league + role, strictly earlier dates)
def brute_percentile(st, k):
    me = st.loc[k]
    ref = st[(st.league_code == me.league_code) & (st.role == me.role) & (st.date < me.date)].raw_value.dropna().to_numpy()
    if len(ref) < MIN_REFERENCE_N:
        return np.nan
    below, equal = (ref < me.raw_value).sum(), (ref == me.raw_value).sum()
    return min(100.0, max(0.0, 100 * (below + equal / 2 - 0.5) / (len(ref) - 1)))

for label, st in (("Stage 1", s1_starters), ("Stage 2", s2_starters_all)):
    idx = st[st.raw_value.notna() & st.role.notna()].sample(300, random_state=42).index
    diffs = []
    for k in idx:
        exp, got = brute_percentile(st, k), st.loc[k, "strength"]
        diffs.append(0.0 if (np.isnan(exp) and np.isnan(got)) else abs(exp - got) if not (np.isnan(exp) or np.isnan(got)) else 99.0)
    record(f"E2 {label} percentile recomputed (300 random starters)", max(diffs) < 1e-6, f"max difference {max(diffs):.2e}")

PASS  E2 Stage 1 percentile recomputed (300 random starters): max difference 0.00e+00


PASS  E2 Stage 2 percentile recomputed (300 random starters): max difference 0.00e+00


In [26]:
# E3: no future information in player values
for label, st, dcol in (("Stage 1", s1_starters, "rating_date"), ("Stage 2", s2_starters_all, "valuation_date")):
    has = st[dcol].notna()
    before = bool((st.loc[has, dcol] < st.loc[has, "date"]).all())
    max_age = st.loc[has, "value_age_days"].max()
    record(f"E3 {label} values dated strictly before the match", before and max_age <= MAX_VALUE_AGE_DAYS,
           f"{has.sum():,} values, all before the match day; oldest {max_age:.0f} days")

PASS  E3 Stage 1 values dated strictly before the match: 536,298 values, all before the match day; oldest 730 days
PASS  E3 Stage 2 values dated strictly before the match: 830,296 values, all before the match day; oldest 730 days


In [27]:
# E4: dataset integrity
target_ok = (result_from_goals(primary.audit_home_goals, primary.audit_away_goals) == primary.target_result).all()
record("E4 Target equals the result of the goals", target_ok, f"{len(primary):,} rows checked")
record("E4 No duplicate matches", not primary.duplicated(["date", "home_team_id", "away_team_id", "league_code"]).any()
       and primary.match_uid.is_unique, "match_uid unique; no repeated date + teams + league")
strength_cols = [c for c in primary.columns if c.endswith("_strength")]
lo, hi = primary[strength_cols].min().min(), primary[strength_cols].max().max()
record("E4 Strength values within 0-100", bool(lo >= 0 and hi <= 100), f"{len(strength_cols)} columns, min {lo}, max {hi}")
record("E4 Form window never larger than 5", int(primary[["home_form_n_L5", "away_form_n_L5"]].max().max()) <= 5, "max = 5")
record("E4 Primary has only the 9 leagues", sorted(primary.league_code.unique()) == sorted(PRIMARY_LEAGUES), f"{primary.league_code.nunique()} leagues")
record("E4 Stage handover without overlap", stage1.date.max() < stage2.date.min(), f"{stage1.date.max()} -> {stage2.date.min()}")

PASS  E4 Target equals the result of the goals: 51,201 rows checked
PASS  E4 No duplicate matches: match_uid unique; no repeated date + teams + league
PASS  E4 Strength values within 0-100: 12 columns, min 0.0, max 100.0
PASS  E4 Form window never larger than 5: max = 5
PASS  E4 Primary has only the 9 leagues: 9 leagues
PASS  E4 Stage handover without overlap: 2016-05-25 -> 2016-07-29


In [28]:
# E5: compare with the dataset used by the ML pipeline (optional)
if EXISTING_PRIMARY and Path(EXISTING_PRIMARY).exists():
    old = pd.read_csv(EXISTING_PRIMARY, low_memory=False).set_index("match_uid").sort_index()
    new = primary.set_index("match_uid").sort_index()
    record("E5 Same 51,201 matches as the ML dataset", old.index.equals(new.index), f"rebuilt {len(new):,} vs ML file {len(old):,}")
    rows = []
    for c in old.columns:
        a, b = new[c], old[c]
        if pd.api.types.is_numeric_dtype(a) and pd.api.types.is_numeric_dtype(b):
            d = (a.astype(float) - b.astype(float)).abs()
            same = ((a.isna() & b.isna()) | (d <= 0.001)).mean()     # 0.001 = rounding tolerance
            rows.append((c, round(same * 100, 3), round(float(d.mean()), 5), round(float(d.max()) if d.notna().any() else 0.0, 4)))
        else:
            same = ((a.isna() & b.isna()) | (a.astype(str) == b.astype(str))).mean()
            rows.append((c, round(same * 100, 3), np.nan, np.nan))
    cmp = pd.DataFrame(rows, columns=["column", "pct_rows_identical", "mean_abs_diff", "max_abs_diff"])
    cmp.to_csv(OUTPUT_DIR / "comparison_with_ML_dataset.csv", index=False)
    n_ident = int((cmp.pct_rows_identical == 100).sum())
    key_cols = ["date", "league_code", "home_team_id", "away_team_id", "audit_home_goals", "audit_away_goals", "target_result",
                "home_points_per_game_L5", "away_points_per_game_L5", "home_goal_diff_avg_L5", "away_goal_diff_avg_L5",
                "home_lineup_continuity", "away_lineup_continuity"]
    key_ok = bool((cmp.set_index("column").loc[key_cols, "pct_rows_identical"] == 100).all())
    record("E5 Target, teams, dates, form and continuity identical to the ML dataset", key_ok, f"{len(key_cols)} key columns, every row")
    max_mean = cmp.mean_abs_diff.max()
    record("E5 All columns agree with the ML dataset", max_mean < 0.05,
           f"{n_ident} of {len(cmp)} columns identical in every row; largest mean difference {max_mean:.4f}")
    print("\nColumns that are not identical in every row:")
    display(cmp[cmp.pct_rows_identical < 100].sort_values("pct_rows_identical"))
    diff_rows = (new.home_xi_strength - old.home_xi_strength).abs() > 0.001
    print("Rows where XI strength differs by more than 0.001, by source and league:")
    print(new[diff_rows].groupby(["data_source", "league_code"]).size().to_string())
    print(f"Median size of those differences: {(new.home_xi_strength - old.home_xi_strength).abs()[diff_rows].median():.4f} points on the 0-100 scale")
else:
    print("E5 skipped: set EXISTING_PRIMARY to the CSV used by the ML pipeline to run this comparison.")

PASS  E5 Same 51,201 matches as the ML dataset: rebuilt 51,201 vs ML file 51,201


PASS  E5 Target, teams, dates, form and continuity identical to the ML dataset: 13 key columns, every row
PASS  E5 All columns agree with the ML dataset: 45 of 61 columns identical in every row; largest mean difference 0.0120

Columns that are not identical in every row:


,column,pct_rows_identical,mean_abs_diff,max_abs_diff
42,home_defence_strength,97.301,0.00037,1.2508
43,away_defence_strength,97.305,0.00035,0.0812
53,home_attack_vs_away_defence,97.363,0.00036,0.0812
54,away_attack_vs_home_defence,97.363,0.00039,1.2508
36,home_xi_strength,97.408,0.00013,0.3328
37,away_xi_strength,97.432,0.00012,0.0296
47,away_top3_strength,97.916,0.00017,0.0609
46,home_top3_strength,97.936,0.00017,0.0619
48,home_top3_gap,97.990,0.00010,0.3328
49,away_top3_gap,98.037,0.00009,0.0403


Rows where XI strength differs by more than 0.001, by source and league:
data_source                    league_code
european_soccer_sqlite_stage1  BE1            1327
Median size of those differences: 0.0032 points on the 0-100 scale


In [29]:
summary = pd.DataFrame(results, columns=["check", "status", "detail"])
summary.to_csv(OUTPUT_DIR / "verification_checks.csv", index=False)
display(summary)
print(f"\n{(summary.status == 'PASS').sum()} of {len(summary)} checks passed | total run time {(time.time() - T0) / 60:.1f} min")

,check,status,detail
0,"E1 Recent form recomputed (1,000 random team-m...",PASS,1000 of 1000 identical
1,E2 Stage 1 percentile recomputed (300 random s...,PASS,max difference 0.00e+00
2,E2 Stage 2 percentile recomputed (300 random s...,PASS,max difference 0.00e+00
3,E3 Stage 1 values dated strictly before the match,PASS,"536,298 values, all before the match day; olde..."
4,E3 Stage 2 values dated strictly before the match,PASS,"830,296 values, all before the match day; olde..."
5,E4 Target equals the result of the goals,PASS,"51,201 rows checked"
6,E4 No duplicate matches,PASS,match_uid unique; no repeated date + teams + l...
7,E4 Strength values within 0-100,PASS,"12 columns, min 0.0, max 100.0"
8,E4 Form window never larger than 5,PASS,max = 5
9,E4 Primary has only the 9 leagues,PASS,9 leagues



14 of 14 checks passed | total run time 1.6 min


---
## Summary
* **Stage 1** and **Stage 2** are built with the same helper functions, so every column means the same thing in both.
* FIFA ratings and market values are put on **one 0-100 scale** by ranking each player within league and role, using only earlier dates.
* The stages are **appended, not joined**, because the two databases use different ID systems.
* The **Primary** file (9 leagues, 51,201 matches) is the input of the ML pipeline. Missing values are left as they are here and filled later inside the ML pipeline using the training period only.
* Part E recomputes the key features independently and compares them with the build, and checks that nothing from the match day or later is used.
* Compared with the file used by the ML pipeline, the rebuild gives the same 51,201 matches and identical targets, teams, dates, form and continuity. The only differences are in the player-strength columns of some Belgian Stage 1 matches (median 0.003 points on the 0-100 scale), plus a few mean-age values and one match whose starter count differs by one. These are far too small to change any model result.